In [14]:
!pip install -q gradio transformers torch reportlab

In [15]:
import gradio as gr
import re
import sqlite3
from datetime import datetime
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

In [16]:
schema = {
    "users": ["id", "name", "email", "signup_date", "age", "country", "status"],
    "orders": ["id", "user_id", "product_name", "amount", "order_date", "status"],
    "products": ["id", "name", "price", "category", "stock"],
    "transactions": ["id", "user_id", "amount", "date", "type"]
}

In [17]:
dangerous_keywords = [
    "drop", "delete", "alter", "truncate",
    "insert", "update", "exec", "execute"
]

def is_safe(text):
    text = text.lower()

    for word in dangerous_keywords:
        if word in text:
            return False

    if ";" in text or "--" in text or "/*" in text:
        return False

    return True

In [18]:
def get_table_name(query):

    query = query.lower()

    if any(word in query for word in ["user", "users", "customer", "account"]):
        return "users"

    if any(word in query for word in ["order", "purchase", "buy"]):
        return "orders"

    if any(word in query for word in ["product", "item", "goods"]):
        return "products"

    if any(word in query for word in ["transaction", "payment", "transfer"]):
        return "transactions"

    return None

In [19]:
def get_columns(query):

    columns = []

    mapping = {
        "name": ["name", "username"],
        "email": ["email", "mail"],
        "signup_date": ["signup", "joined", "registered"],
        "age": ["age"],
        "amount": ["amount", "price", "cost"],
        "status": ["status"],
        "country": ["country", "location"]
    }

    query = query.lower()

    for column, keywords in mapping.items():
        for word in keywords:
            if word in query:
                columns.append(column)

    if not columns:
        columns.append("*")

    return columns

In [20]:
def extract_conditions(query):

    query = query.lower()
    conditions = []

    # amount condition
    amount_match = re.search(r'greater than (\d+)', query)
    if amount_match:
        conditions.append(f"amount > {amount_match.group(1)}")

    # last week
    if "last week" in query:
        conditions.append("DATE(date) >= DATE('now','-7 days')")

    # last month
    if "last month" in query:
        conditions.append("DATE(signup_date) >= DATE('now','-1 month')")

    return conditions

In [21]:
def generate_sql(query):

    if not is_safe(query):
        return "⚠️ Unsafe query detected!"

    table = get_table_name(query)

    if not table:
        return "Table not detected."

    columns = get_columns(query)
    conditions = extract_conditions(query)

    column_string = ", ".join(columns)

    sql = f"SELECT {column_string} FROM {table}"

    if conditions:
        sql += " WHERE " + " AND ".join(conditions)

    sql += " LIMIT 10"

    return sql

In [22]:
def export_pdf(query, sql):

    filename = "query_report.pdf"

    c = canvas.Canvas(filename, pagesize=letter)

    c.drawString(100, 750, "SQL Query Generator Report")
    c.drawString(100, 720, "User Request:")
    c.drawString(100, 700, query)

    c.drawString(100, 660, "Generated SQL:")
    c.drawString(100, 640, sql)

    c.save()

    return filename

In [23]:
def process_query(user_input):

    sql = generate_sql(user_input)

    return sql


demo = gr.Interface(
    fn=process_query,
    inputs=gr.Textbox(label="Enter your request"),
    outputs=gr.Textbox(label="Generated SQL Query"),
    title="SQL Query Generator",
    description="Convert Natural Language to SQL"
)

In [24]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://77c69348c286255a02.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
